# Quantum Force - Bronze to Gold

Assumption: Bronze Delta tables already exist in the configured Bronze schema (loaded by bronze_load).
There is no Silver layer: cleansing and validation run in memory, and results are written
directly to Gold dimensions/bridges/facts, a defect knowledge base, DQ audit results,
and reconciliation outputs.

Set overwrite=true to replace existing Gold tables.
Set overwrite=false to create Gold tables only when they do not already exist.

In [0]:
# CELL 1 - IMPORTS, WIDGETS AND CONFIGURATION

from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import datetime, timezone

# Aligned with the bronze_load notebook:
#   source_path   = /Volumes/team_8/quantum_force/quantum_force
#   bronze target = team_8.quantum_force.<json_file_stem_lowercase>
dbutils.widgets.text("catalog", "team_8", "Unity Catalog")
dbutils.widgets.text("bronze_schema", "quantum_force", "Bronze schema")
dbutils.widgets.text("gold_schema", "quantum_force_gold", "Gold schema")
dbutils.widgets.dropdown("overwrite", "true", ["true", "false"], "Overwrite Gold tables")

CATALOG = dbutils.widgets.get("catalog").strip()
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema").strip()
GOLD_SCHEMA = dbutils.widgets.get("gold_schema").strip()
OVERWRITE = dbutils.widgets.get("overwrite").lower() == "true"
WRITE_MODE = "overwrite" if OVERWRITE else "errorifexists"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

# Audit columns added by bronze_load; stripped before Gold modelling
BRONZE_AUDIT_COLS = ["_source_file", "_bronze_load_timestamp"]

def qname(schema, table):
    return f"`{CATALOG}`.`{schema}`.`{table}`"

def table_exists(schema, table):
    return spark.catalog.tableExists(qname(schema, table))

# Bronze schema must already exist (created and loaded by bronze_load)
if not spark.catalog.databaseExists(f"{CATALOG}.{BRONZE_SCHEMA}"):
    raise ValueError(
        f"Bronze schema {CATALOG}.{BRONZE_SCHEMA} not found. "
        "Run the bronze_load notebook first or fix the catalog/bronze_schema widgets."
    )

# Gold must be separate from Bronze: Bronze already holds dim_calendar / dim_time
if BRONZE_SCHEMA == GOLD_SCHEMA:
    raise ValueError("Gold schema must differ from the Bronze schema.")

try:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{GOLD_SCHEMA}`")
except Exception as e:
    raise PermissionError(
        f"Cannot create schema {CATALOG}.{GOLD_SCHEMA}. Request CREATE SCHEMA on "
        f"catalog {CATALOG} or point the widget to an existing schema. "
        f"{str(e).split('JVM stacktrace')[0]}"
    )

print(f"Bronze : {CATALOG}.{BRONZE_SCHEMA}")
print(f"Gold   : {CATALOG}.{GOLD_SCHEMA}")
print(f"Overwrite existing Gold tables: {OVERWRITE}")
print(f"Run ID: {RUN_ID}")

Bronze : team_8.quantum_force
Gold   : team_8.quantum_force_gold
Overwrite existing Gold tables: True
Run ID: 20261118091143


In [0]:
# CELL 2 - BRONZE SOURCE TABLES AND PRE-FLIGHT CHECKS

FILES = [
    "projects", "teams", "roles", "team_roles", "developers", "people",
    "team_members", "user_team_access", "epics", "features", "sprints",
    "project_sprint_bridge", "dim_calendar", "dim_time",
    "issues_enriched", "issue_history"
]

# bronze_load names each table after the lowercase JSON file stem
available = sorted(
    t.name.lower() for t in spark.catalog.listTables(f"{CATALOG}.{BRONZE_SCHEMA}")
)
missing = [t for t in FILES if t not in available]
if missing:
    raise ValueError(
        f"Missing Bronze tables in {CATALOG}.{BRONZE_SCHEMA}: {missing}. "
        f"Available tables: {available}. Run the bronze_load notebook first."
    )

# Strip bronze_load audit columns so they don't leak into Gold models
b = {
    name: spark.table(qname(BRONZE_SCHEMA, name)).drop(*BRONZE_AUDIT_COLS)
    for name in FILES
}

bronze_counts = {}
for name, df in b.items():
    count = df.count()
    bronze_counts[name] = count
    print(f"{name:<28} {count:>12,}")

empty_tables = [name for name, count in bronze_counts.items() if count == 0]
if empty_tables:
    raise ValueError(f"These Bronze tables are empty: {empty_tables}")

print("Bronze pre-flight checks passed.")

def bronze(name):
    # Always read the latest Bronze version, so rows merged after this cell ran
    # (e.g. the active sprint MERGE in bronze_load) are picked up by later cells
    return spark.table(qname(BRONZE_SCHEMA, name)).drop(*BRONZE_AUDIT_COLS)

for _t in ("issues_enriched", "sprints", "project_sprint_bridge"):
    _last = spark.sql(f"DESCRIBE HISTORY {qname(BRONZE_SCHEMA, _t)} LIMIT 1").first()
    print(f"Bronze {_t:<24} version={_last['version']} op={_last['operation']} at={_last['timestamp']}")

projects                                5
teams                                   8
roles                                   7
team_roles                             10
developers                          1,190
people                              1,250
team_members                           60
user_team_access                        8
epics                                 100
features                              300
sprints                               400
project_sprint_bridge               1,969
dim_calendar                        5,960
dim_time                           86,400
issues_enriched                    20,177
issue_history                      54,309
Bronze pre-flight checks passed.


In [0]:
# CELL 3 - REUSABLE HELPERS AND DATA QUALITY LOGGING

dq_log = []

def assert_columns(df, required_columns, label):
    missing_columns = [c for c in required_columns if c not in df.columns]
    if missing_columns:
        raise ValueError(f"{label} is missing required columns: {missing_columns}")

def assert_unique(df, columns, label):
    cols = list(columns) if isinstance(columns, (list, tuple)) else [columns]
    duplicates = (
        df.groupBy(*cols).count()
          .filter(F.col("count") > 1)
          .count()
    )
    if duplicates:
        raise ValueError(f"{label}: found {duplicates} duplicate key groups for {cols}")

def assert_not_null(df, columns, label):
    for c in columns:
        bad = df.filter(F.col(c).isNull()).limit(1).count()
        if bad:
            raise ValueError(f"{label}: mandatory column {c} contains NULL")

def assert_no_orphans(child, child_col, parent, parent_col, label):
    child_keys = child.select(F.col(child_col).cast("string").alias("_key")) \
                       .filter(F.col("_key").isNotNull())
    parent_keys = parent.select(F.col(parent_col).cast("string").alias("_key")) \
                         .filter(F.col("_key").isNotNull()).distinct()
    orphan_count = child_keys.join(parent_keys, "_key", "left_anti").count()
    if orphan_count:
        raise ValueError(f"{label}: found {orphan_count} orphan rows")

def log_check(table_name, check_name, severity, failed_rows, detail=""):
    status = (
        "PASS" if failed_rows == 0
        else "FAIL" if severity == "error"
        else "WARN"
    )
    dq_log.append((
        RUN_ID, table_name, check_name, severity, int(failed_rows),
        status, detail, datetime.now(timezone.utc)
    ))
    print(
        f"[{status}] {table_name}.{check_name}: "
        f"failed_rows={failed_rows} {detail}"
    )
    return status

def validate_for_gold(name, df, pk=None, not_null=(), min_rows=1):
    errors = []
    row_count = df.count()

    if log_check(name, "minimum_row_count", "error",
                 0 if row_count >= min_rows else 1,
                 f"rows={row_count}") == "FAIL":
        errors.append("minimum_row_count")

    if pk:
        keys = list(pk) if isinstance(pk, (list, tuple)) else [pk]
        assert_columns(df, keys, name)
        duplicate_rows = row_count - df.select(*keys).distinct().count()
        if log_check(name, f"unique_key:{','.join(keys)}", "error",
                     duplicate_rows) == "FAIL":
            errors.append("unique_key")

        not_null = tuple(keys) + tuple(not_null)

    for c in dict.fromkeys(not_null):
        assert_columns(df, [c], name)
        bad = df.filter(F.col(c).isNull()).count()
        if log_check(name, f"not_null:{c}", "error", bad) == "FAIL":
            errors.append(f"not_null:{c}")

    return errors

def flush_dq_log():
    if not dq_log:
        return

    log_schema = """
        run_id STRING,
        table_name STRING,
        check_name STRING,
        severity STRING,
        failed_rows LONG,
        status STRING,
        detail STRING,
        checked_at TIMESTAMP
    """
    log_df = spark.createDataFrame(dq_log, schema=log_schema)

    target = qname(GOLD_SCHEMA, "dq_validation_results")
    if spark.catalog.tableExists(target):
        log_df.write.format("delta").mode("append").saveAsTable(target)
    else:
        log_df.write.format("delta").mode("overwrite").saveAsTable(target)

    print(f"Persisted {len(dq_log)} DQ validation records.")
    dq_log.clear()

def save_delta(df, schema, name, comment=None):
    target = qname(schema, name)

    if not OVERWRITE and spark.catalog.tableExists(target):
        raise ValueError(
            f"Target already exists and overwrite=false: {target}"
        )

    output = df.withColumn(
        "_load_run_id", F.lit(RUN_ID)
    ).withColumn(
        "_load_timestamp", F.current_timestamp()
    )

    (output.write.format("delta")
        .mode(WRITE_MODE)
        .option("overwriteSchema", "true")
        .saveAsTable(target))

    if comment:
        safe_comment = comment.replace("'", "''")
        spark.sql(f"COMMENT ON TABLE {target} IS '{safe_comment}'")

    loaded = spark.table(target).count()
    print(f"LOADED {target}: {loaded:,} rows")
    return loaded

In [0]:
# CELL 4 - CLEANSING TRANSFORMATIONS (IN MEMORY, NO SILVER LAYER)

issues_raw = bronze("issues_enriched")
history_raw = bronze("issue_history")

assert_columns(
    issues_raw,
    [
        "issue_id", "issue_key", "project_id", "team_id",
        "assignee_person_id", "sprint_id", "epic_id", "feature_id",
        "created", "updated", "resolution_date", "story_points",
        "cycle_time_days", "is_resolved", "status", "reporter", "developer_id"
    ],
    "issues_enriched"
)
assert_columns(
    history_raw,
    ["history_id", "issue_id", "status", "status_changed_at", "sequence_number"],
    "issue_history"
)

def normalize_nullable_string(column_name):
    value = F.trim(F.col(column_name).cast("string"))
    return F.when(
        value.isNull() | (F.lower(value).isin("", "null", "none", "nan")),
        F.lit(None).cast("string")
    ).otherwise(value)

issues_s = (
    issues_raw
    .withColumn("issue_id", F.col("issue_id").cast("string"))
    .withColumn("issue_key", normalize_nullable_string("issue_key"))
    .withColumn("reporter", normalize_nullable_string("reporter"))
    .withColumn("assignee_person_id",
                normalize_nullable_string("assignee_person_id"))
    .withColumn("developer_id", normalize_nullable_string("developer_id"))
    .withColumn("created_ts", F.to_timestamp("created"))
    .withColumn("updated_ts", F.to_timestamp("updated"))
    .withColumn("resolution_ts", F.to_timestamp("resolution_date"))
    .withColumn("story_points", F.col("story_points").cast("double"))
    .withColumn("cycle_time_days", F.col("cycle_time_days").cast("double"))
    .withColumn("is_resolved",
                F.coalesce(F.col("is_resolved").cast("boolean"), F.lit(False)))
)

history_s = (
    history_raw
    .withColumn("history_id", F.col("history_id").cast("string"))
    .withColumn("issue_id", F.col("issue_id").cast("string"))
    .withColumn("status_changed_ts", F.to_timestamp("status_changed_at"))
    .withColumn("sequence_number", F.col("sequence_number").cast("int"))
)

# Mandatory issue identifiers and project association.
# Sprint, epic, feature and assignee can legitimately be NULL.
assert_not_null(
    issues_s,
    ["issue_id", "issue_key", "project_id"],
    "Cleansed issues"
)
assert_unique(issues_s, "issue_id", "Cleansed issues")
assert_unique(issues_s, "issue_key", "Cleansed issues")

assert_not_null(history_s, ["history_id", "issue_id"], "Cleansed history")
assert_unique(history_s, "history_id", "Cleansed history")

In [0]:
# CELL 5 - REFERENTIAL AND HIERARCHY VALIDATION

assert_no_orphans(
    issues_s, "project_id", bronze("projects"), "project_id",
    "issues -> projects"
)
assert_no_orphans(
    issues_s, "team_id", bronze("teams"), "team_id",
    "issues -> teams"
)
assert_no_orphans(
    issues_s, "assignee_person_id", bronze("people"), "person_id",
    "issues -> people"
)
assert_no_orphans(
    issues_s, "developer_id", bronze("developers"), "developer_id",
    "issues -> developers"
)
assert_no_orphans(
    issues_s, "sprint_id", bronze("sprints"), "sprint_id",
    "issues -> sprints"
)
assert_no_orphans(
    issues_s, "epic_id", bronze("epics"), "epic_id",
    "issues -> epics"
)
assert_no_orphans(
    issues_s, "feature_id", bronze("features"), "feature_id",
    "issues -> features"
)
assert_no_orphans(
    history_s, "issue_id", issues_s, "issue_id",
    "history -> issues"
)

# Validate feature/epic/project consistency when a feature is assigned.
feature_lookup = bronze("features").select(
    F.col("feature_id").cast("string").alias("_feature_id"),
    F.col("epic_id").cast("string").alias("_feature_epic_id"),
    F.col("project_id").cast("string").alias("_feature_project_id")
)

issue_hierarchy = (
    issues_s.alias("i")
    .join(
        feature_lookup.alias("f"),
        F.col("i.feature_id").cast("string") == F.col("f._feature_id"),
        "left"
    )
)

bad_issue_hierarchy = issue_hierarchy.filter(
    F.col("i.feature_id").isNotNull() &
    (
        (F.col("i.epic_id").cast("string") != F.col("f._feature_epic_id")) |
        (F.col("i.project_id").cast("string") != F.col("f._feature_project_id"))
    )
).count()

if bad_issue_hierarchy:
    raise ValueError(
        f"Issue-feature hierarchy mismatch: {bad_issue_hierarchy} rows"
    )

epic_lookup = bronze("epics").select(
    F.col("epic_id").cast("string").alias("_epic_id"),
    F.col("project_id").cast("string").alias("_epic_project_id")
)

feature_hierarchy = (
    bronze("features").alias("f")
    .join(
        epic_lookup.alias("e"),
        F.col("f.epic_id").cast("string") == F.col("e._epic_id"),
        "left"
    )
)

bad_feature_hierarchy = feature_hierarchy.filter(
    F.col("f.epic_id").isNotNull() &
    (F.col("f.project_id").cast("string") != F.col("e._epic_project_id"))
).count()

if bad_feature_hierarchy:
    raise ValueError(
        f"Feature-epic project mismatch: {bad_feature_hierarchy} rows"
    )

print("Referential and hierarchy checks passed.")

Referential and hierarchy checks passed.


In [0]:
# CELL 7 - BUILD AND LOAD GOLD DIMENSIONS

gold_dimensions = {
    "dim_project": (
        bronze("projects").select(
            "project_id", "project_key", "project_name", "description",
            "project_type", "source_project_key", "data_type"
        ),
        "Jira project dimension",
        "project_id"
    ),
    "dim_team": (
        bronze("teams").select(
            "team_id", "team_name", "team_type", "description", "data_type"
        ),
        "Team dimension",
        "team_id"
    ),
    "dim_role": (
        bronze("roles").select(
            "role_id", "role_name", "description", "data_type"
        ),
        "Role dimension",
        "role_id"
    ),
    "dim_person": (
        bronze("people").select(
            "person_id", "person_name", "team_id", "team", "role_id",
            "role_name", "person_type", "developer_id", "data_type"
        ),
        "People and developer dimension",
        "person_id"
    ),
    "dim_epic": (
        bronze("epics").select(
            "epic_id", "epic_name", "project_id", "project_key",
            "status", "created_date", "data_type"
        ),
        "Jira epic dimension",
        "epic_id"
    ),
    "dim_feature": (
        bronze("features").select(
            "feature_id", "feature_name", "epic_id",
            "project_id", "status", "data_type"
        ),
        "Jira feature dimension",
        "feature_id"
    ),
    "dim_sprint": (
        bronze("sprints").select(
            "sprint_id", "sprint_name", "sprint_number", "start_date",
            "end_date", "duration_days", "sprint_type", "status",
            "project_id", "start_date_key", "end_date_key", "data_type"
        ),
        "Jira sprint dimension",
        "sprint_id"
    ),
    "dim_calendar": (
        bronze("dim_calendar"),  # audit columns already stripped in bronze()
        "Calendar dimension",
        "date_key"
    ),
    "dim_time": (
        bronze("dim_time"),  # audit columns already stripped in bronze()
        "Time dimension",
        "time_key"
    )
}

for name, (df, comment, pk) in gold_dimensions.items():
    errors = validate_for_gold(name, df, pk=pk)
    if errors:
        flush_dq_log()
        raise ValueError(f"Gold dimension validation failed for {name}: {errors}")
    save_delta(df, GOLD_SCHEMA, name, comment)

flush_dq_log()

[PASS] dim_project.minimum_row_count: failed_rows=0 rows=5
[PASS] dim_project.unique_key:project_id: failed_rows=0 
[PASS] dim_project.not_null:project_id: failed_rows=0 
LOADED `team_8`.`quantum_force_gold`.`dim_project`: 5 rows
[PASS] dim_team.minimum_row_count: failed_rows=0 rows=8
[PASS] dim_team.unique_key:team_id: failed_rows=0 
[PASS] dim_team.not_null:team_id: failed_rows=0 
LOADED `team_8`.`quantum_force_gold`.`dim_team`: 8 rows
[PASS] dim_role.minimum_row_count: failed_rows=0 rows=7
[PASS] dim_role.unique_key:role_id: failed_rows=0 
[PASS] dim_role.not_null:role_id: failed_rows=0 
LOADED `team_8`.`quantum_force_gold`.`dim_role`: 7 rows
[PASS] dim_person.minimum_row_count: failed_rows=0 rows=1250
[PASS] dim_person.unique_key:person_id: failed_rows=0 
[PASS] dim_person.not_null:person_id: failed_rows=0 
LOADED `team_8`.`quantum_force_gold`.`dim_person`: 1,250 rows
[PASS] dim_epic.minimum_row_count: failed_rows=0 rows=100
[PASS] dim_epic.unique_key:epic_id: failed_rows=0 
[PASS]

In [0]:
# CELL 8 - BUILD AND LOAD GOLD BRIDGE TABLES

gold_bridges = {
    "bridge_project_sprint": (
        bronze("project_sprint_bridge"),
        "Project to sprint many-to-many bridge",
        ["project_id", "sprint_id"]
    ),
    "bridge_team_role": (
        bronze("team_roles"),
        "Team to role bridge",
        ["team_id", "role_id"]
    ),
    "bridge_user_team_access": (
        bronze("user_team_access"),
        "User to team access bridge",
        "access_id"
    )
}

bridge_fk_checks = {
    "bridge_project_sprint": [
        ("project_id", bronze("projects"), "project_id"),
        ("sprint_id", bronze("sprints"), "sprint_id")
    ],
    "bridge_team_role": [
        ("team_id", bronze("teams"), "team_id"),
        ("role_id", bronze("roles"), "role_id")
    ],
    "bridge_user_team_access": [
        ("team_id", bronze("teams"), "team_id"),
        ("role_id", bronze("roles"), "role_id")
    ]
}

for name, (df, comment, pk) in gold_bridges.items():
    errors = validate_for_gold(name, df, pk=pk)
    if errors:
        flush_dq_log()
        raise ValueError(f"Gold bridge validation failed for {name}: {errors}")

    for child_col, parent_df, parent_col in bridge_fk_checks[name]:
        child_keys = df.select(
            F.col(child_col).cast("string").alias("_key")
        ).filter(F.col("_key").isNotNull())
        parent_keys = parent_df.select(
            F.col(parent_col).cast("string").alias("_key")
        ).filter(F.col("_key").isNotNull()).distinct()

        orphan_count = child_keys.join(
            parent_keys, "_key", "left_anti"
        ).count()

        log_check(name, f"fk:{child_col}", "error", orphan_count)

        if orphan_count:
            flush_dq_log()
            raise ValueError(
                f"Bridge {name} has {orphan_count} orphan {child_col} values"
            )

    save_delta(df, GOLD_SCHEMA, name, comment)

flush_dq_log()

[PASS] bridge_project_sprint.minimum_row_count: failed_rows=0 rows=1969
[PASS] bridge_project_sprint.unique_key:project_id,sprint_id: failed_rows=0 
[PASS] bridge_project_sprint.not_null:project_id: failed_rows=0 
[PASS] bridge_project_sprint.not_null:sprint_id: failed_rows=0 
[PASS] bridge_project_sprint.fk:project_id: failed_rows=0 
[PASS] bridge_project_sprint.fk:sprint_id: failed_rows=0 
LOADED `team_8`.`quantum_force_gold`.`bridge_project_sprint`: 1,969 rows
[PASS] bridge_team_role.minimum_row_count: failed_rows=0 rows=10
[PASS] bridge_team_role.unique_key:team_id,role_id: failed_rows=0 
[PASS] bridge_team_role.not_null:team_id: failed_rows=0 
[PASS] bridge_team_role.not_null:role_id: failed_rows=0 
[PASS] bridge_team_role.fk:team_id: failed_rows=0 
[PASS] bridge_team_role.fk:role_id: failed_rows=0 
LOADED `team_8`.`quantum_force_gold`.`bridge_team_role`: 10 rows
[PASS] bridge_user_team_access.minimum_row_count: failed_rows=0 rows=8
[PASS] bridge_user_team_access.unique_key:access

In [0]:
# CELL 9 - BUILD AND LOAD GOLD FACT ISSUE

people_lookup = bronze("people").select(
    F.col("person_id").cast("string").alias("_person_id"),
    F.col("person_name").alias("assignee_person_name"),
    F.col("person_type").alias("assignee_person_type")
)

issues_gold = (
    issues_s  # cleansed in memory (no Silver layer)
    .join(
        people_lookup,
        F.col("assignee_person_id") == F.col("_person_id"),
        "left"
    )
    .drop("_person_id")
    .withColumnRenamed("resolution_ts", "resolution_date_ts")
)

# Validate issue fact grain and mandatory fields.
errors = validate_for_gold(
    "fact_issue",
    issues_gold,
    pk="issue_id",
    not_null=["issue_key", "project_id", "created_ts", "status"]
)
if errors:
    flush_dq_log()
    raise ValueError(f"fact_issue validation failed: {errors}")

# Business-rule checks.
negative_story_points = issues_gold.filter(
    F.col("story_points") < 0
).count()
negative_cycle_time = issues_gold.filter(
    F.col("cycle_time_days") < 0
).count()

log_check(
    "fact_issue", "negative_story_points", "error",
    negative_story_points
)
log_check(
    "fact_issue", "negative_cycle_time", "error",
    negative_cycle_time
)

if negative_story_points or negative_cycle_time:
    flush_dq_log()
    raise ValueError("fact_issue has negative story points or cycle time.")

save_delta(
    issues_gold,
    GOLD_SCHEMA,
    "fact_issue",
    "One row per Jira issue, enriched with assignee person attributes"
)
flush_dq_log()

[PASS] fact_issue.minimum_row_count: failed_rows=0 rows=20177
[PASS] fact_issue.unique_key:issue_id: failed_rows=0 
[PASS] fact_issue.not_null:issue_id: failed_rows=0 
[PASS] fact_issue.not_null:issue_key: failed_rows=0 
[PASS] fact_issue.not_null:project_id: failed_rows=0 
[PASS] fact_issue.not_null:created_ts: failed_rows=0 
[PASS] fact_issue.not_null:status: failed_rows=0 
[PASS] fact_issue.negative_story_points: failed_rows=0 
[PASS] fact_issue.negative_cycle_time: failed_rows=0 
LOADED `team_8`.`quantum_force_gold`.`fact_issue`: 20,177 rows
Persisted 9 DQ validation records.


In [0]:
# CELL 10 - BUILD AND LOAD GOLD STATUS HISTORY FACT

history_gold = history_s  # cleansed in memory (no Silver layer)

errors = validate_for_gold(
    "fact_status_transition",
    history_gold,
    pk="history_id",
    not_null=["issue_id", "status"]
)
if errors:
    flush_dq_log()
    raise ValueError(f"fact_status_transition validation failed: {errors}")

assert_no_orphans(
    history_gold,
    "issue_id",
    issues_gold,
    "issue_id",
    "status history -> issue fact"
)

save_delta(
    history_gold,
    GOLD_SCHEMA,
    "fact_status_transition",
    "One row per simulated Jira status transition"
)
flush_dq_log()

[PASS] fact_status_transition.minimum_row_count: failed_rows=0 rows=54309
[PASS] fact_status_transition.unique_key:history_id: failed_rows=0 
[PASS] fact_status_transition.not_null:history_id: failed_rows=0 
[PASS] fact_status_transition.not_null:issue_id: failed_rows=0 
[PASS] fact_status_transition.not_null:status: failed_rows=0 
LOADED `team_8`.`quantum_force_gold`.`fact_status_transition`: 54,309 rows
Persisted 5 DQ validation records.


In [0]:
# CELL 11 - TEAM CAPACITY AND SPRINT DELIVERY FACT

developers = bronze("developers")

assert_columns(
    developers,
    ["team_id", "developer_id", "capacity_points_per_sprint"],
    "developers"
)

# Calculate capacity independently at team grain.
# This avoids multiplying capacity by the number of issue rows.
developer_capacity = (
    developers
    .groupBy("team_id", "developer_id")
    .agg(
        F.max(
            F.col("capacity_points_per_sprint").cast("double")
        ).alias("developer_capacity_points")
    )
)

team_capacity = (
    developer_capacity
    .groupBy("team_id")
    .agg(
        F.sum("developer_capacity_points").alias(
            "team_capacity_points_per_sprint"
        ),
        F.countDistinct("developer_id").alias("developer_count")
    )
)

fi = spark.table(qname(GOLD_SCHEMA, "fact_issue"))

sprint_delivery = (
    fi.filter(F.col("sprint_id").isNotNull())
    .groupBy("project_id", "sprint_id", "team_id")
    .agg(
        F.countDistinct("issue_id").alias("assigned_issue_count"),
        F.sum("story_points").alias("assigned_points_proxy"),
        F.sum(
            F.when(F.col("is_resolved") == True, F.col("story_points"))
             .otherwise(F.lit(0.0))
        ).alias("completed_points"),
        F.countDistinct(
            F.when(F.col("is_resolved") == True, F.col("issue_id"))
        ).alias("completed_issue_count"),
        F.countDistinct(
            F.when(F.col("is_resolved") != True, F.col("issue_id"))
        ).alias("open_issue_count")
    )
    .join(team_capacity, "team_id", "left")
    .withColumn(
        "spillover_issue_count",
        F.col("open_issue_count")
    )
    .withColumn(
        "commitment_measure_type",
        F.lit("Assigned points proxy; no sprint-start snapshot")
    )
    .withColumn(
        "completion_rate",
        F.when(
            F.col("assigned_points_proxy") > 0,
            F.round(
                F.col("completed_points") /
                F.col("assigned_points_proxy"),
                4
            )
        )
    )
)

errors = validate_for_gold(
    "fact_sprint_delivery",
    sprint_delivery,
    pk=["project_id", "sprint_id", "team_id"],
    not_null=["project_id", "sprint_id"]
)
if errors:
    flush_dq_log()
    raise ValueError(f"fact_sprint_delivery validation failed: {errors}")

save_delta(
    sprint_delivery,
    GOLD_SCHEMA,
    "fact_sprint_delivery",
    "Sprint delivery metrics by project, sprint and team"
)
flush_dq_log()

[PASS] fact_sprint_delivery.minimum_row_count: failed_rows=0 rows=7916
[PASS] fact_sprint_delivery.unique_key:project_id,sprint_id,team_id: failed_rows=0 
[PASS] fact_sprint_delivery.not_null:project_id: failed_rows=0 
[PASS] fact_sprint_delivery.not_null:sprint_id: failed_rows=0 
[PASS] fact_sprint_delivery.not_null:team_id: failed_rows=0 
LOADED `team_8`.`quantum_force_gold`.`fact_sprint_delivery`: 7,916 rows
Persisted 5 DQ validation records.


In [0]:
# CELL 12 - DEFECT KNOWLEDGE BASE

# Summary is retained as the source's free-text defect description.
kedb = (
    fi.filter(
        (F.lower(F.col("issue_type")) == "bug") &
        F.col("summary").isNotNull()
    )
    .select(
        "issue_id", "issue_key", "project_id", "project_key",
        "team_id", "team", "assignee_person_id", "developer_id",
        "summary", "issue_type", "priority", "story_points",
        "status", "is_resolved", "resolution",
        "resolution_date", "cycle_time_days",
        "epic_id", "feature_id", "sprint_id", "data_type"
    )
)

errors = validate_for_gold(
    "gold_defect_knowledge",
    kedb,
    pk="issue_id",
    not_null=["issue_key", "summary"]
)
if errors:
    flush_dq_log()
    raise ValueError(f"Defect knowledge validation failed: {errors}")

save_delta(
    kedb,
    GOLD_SCHEMA,
    "gold_defect_knowledge",
    "Knowledge base of Jira bug issues and their descriptive summaries"
)
flush_dq_log()

[PASS] gold_defect_knowledge.minimum_row_count: failed_rows=0 rows=9161
[PASS] gold_defect_knowledge.unique_key:issue_id: failed_rows=0 
[PASS] gold_defect_knowledge.not_null:issue_id: failed_rows=0 
[PASS] gold_defect_knowledge.not_null:issue_key: failed_rows=0 
[PASS] gold_defect_knowledge.not_null:summary: failed_rows=0 
LOADED `team_8`.`quantum_force_gold`.`gold_defect_knowledge`: 9,161 rows
Persisted 5 DQ validation records.


In [0]:
# CELL 13 - GOLD TABLE INVENTORY AND ROW COUNTS

GOLD_TABLES = [
    "dim_project", "dim_team", "dim_role", "dim_person",
    "dim_epic", "dim_feature", "dim_sprint", "dim_calendar",
    "dim_time", "bridge_project_sprint", "bridge_team_role",
    "bridge_user_team_access", "fact_issue",
    "fact_status_transition", "fact_sprint_delivery",
    "gold_defect_knowledge"
]

gold_summary = []
for name in GOLD_TABLES:
    if table_exists(GOLD_SCHEMA, name):
        count = spark.table(qname(GOLD_SCHEMA, name)).count()
        gold_summary.append((name, count, "EXISTS"))
    else:
        gold_summary.append((name, 0, "MISSING"))

summary_df = spark.createDataFrame(
    gold_summary,
    ["table_name", "row_count", "table_status"]
)
display(summary_df)

table_name,row_count,table_status
dim_project,5,EXISTS
dim_team,8,EXISTS
dim_role,7,EXISTS
dim_person,1250,EXISTS
dim_epic,100,EXISTS
dim_feature,300,EXISTS
dim_sprint,400,EXISTS
dim_calendar,5960,EXISTS
dim_time,86400,EXISTS
bridge_project_sprint,1969,EXISTS


In [0]:
# CELL 14 - BRONZE-TO-GOLD RECONCILIATION

bronze_issue_count = bronze("issues_enriched").count()
gold_issue_count = spark.table(qname(GOLD_SCHEMA, "fact_issue")).count()

bronze_history_count = bronze("issue_history").count()
gold_history_count = spark.table(
    qname(GOLD_SCHEMA, "fact_status_transition")
).count()

bronze_sprint_issue_count = (
    bronze("issues_enriched")
    .filter(F.col("sprint_id").isNotNull())
    .select("issue_id").distinct().count()
)

gold_sprint_issue_count = (
    spark.table(qname(GOLD_SCHEMA, "fact_sprint_delivery"))
    .agg(F.sum("assigned_issue_count").alias("n"))
    .first()["n"] or 0
)

reconciliation = [
    ("Issue row count", bronze_issue_count, gold_issue_count),
    ("Status history row count", bronze_history_count, gold_history_count),
    ("Issues assigned to sprints", bronze_sprint_issue_count, gold_sprint_issue_count)
]

recon_df = spark.createDataFrame(
    reconciliation,
    ["check_name", "bronze_value", "gold_value"]
).withColumn(
    "match",
    F.col("bronze_value") == F.col("gold_value")
)

display(recon_df)

# Key-level checks: every Bronze key must exist in Gold (catches merged rows not loaded)
key_checks = [
    ("issues_enriched -> fact_issue", "issues_enriched", "fact_issue", ["issue_id"]),
    ("sprints -> dim_sprint", "sprints", "dim_sprint", ["sprint_id"]),
    ("project_sprint_bridge -> bridge_project_sprint", "project_sprint_bridge",
     "bridge_project_sprint", ["project_id", "sprint_id"]),
]
key_results = []
for label, b_table, g_table, keys in key_checks:
    b_keys = bronze(b_table).select(*[F.col(k).cast("string").alias(k) for k in keys]).distinct()
    g_keys = spark.table(qname(GOLD_SCHEMA, g_table)).select(
        *[F.col(k).cast("string").alias(k) for k in keys]).distinct()
    missing = b_keys.join(g_keys, keys, "left_anti")
    n_missing = missing.count()
    key_results.append((label, b_keys.count(), n_missing))
    if n_missing:
        print(f"{label}: {n_missing} Bronze keys missing in Gold, e.g.")
        display(missing.limit(20))

key_df = spark.createDataFrame(key_results, "check_name STRING, bronze_keys LONG, missing_in_gold LONG")
display(key_df)

recon_failures = (
    recon_df.filter(~F.col("match")).count()
    + key_df.filter(F.col("missing_in_gold") > 0).count()
)

check_name,bronze_value,gold_value,match
Issue row count,20177,20177,true
Status history row count,54309,54309,true
Issues assigned to sprints,20177,20177,true


In [0]:
# CELL 15 - DQ RUN SUMMARY AND FINAL STATUS

dq_target = qname(GOLD_SCHEMA, "dq_validation_results")

if spark.catalog.tableExists(dq_target):
    run_results = (
        spark.table(dq_target)
        .filter(F.col("run_id") == RUN_ID)
        .orderBy("status", "table_name", "check_name")
    )
    display(run_results)

    display(
        run_results.groupBy("status")
        .count()
        .orderBy("status")
    )

if recon_failures:
    raise ValueError(
        f"Gold reconciliation failed: {recon_failures} mismatch(es). "
        "Review the reconciliation output."
    )

print("=" * 70)
print("QUANTUM FORCE BRONZE-TO-GOLD BUILD COMPLETED")
print(f"Run ID: {RUN_ID}")
print(f"Gold schema: {CATALOG}.{GOLD_SCHEMA}")
print("=" * 70)
print("Next step: execute the Quantum Force KPI SQL pack.")

run_id,table_name,check_name,severity,failed_rows,status,detail,checked_at
20261118091143,bridge_project_sprint,fk:project_id,error,0,PASS,,2026-11-18T09:13:11.529Z
20261118091143,bridge_project_sprint,fk:sprint_id,error,0,PASS,,2026-11-18T09:13:12.165Z
20261118091143,bridge_project_sprint,minimum_row_count,error,0,PASS,rows=1969,2026-11-18T09:13:09.747Z
20261118091143,bridge_project_sprint,not_null:project_id,error,0,PASS,,2026-11-18T09:13:10.579Z
20261118091143,bridge_project_sprint,not_null:sprint_id,error,0,PASS,,2026-11-18T09:13:10.861Z
20261118091143,bridge_project_sprint,"unique_key:project_id,sprint_id",error,0,PASS,,2026-11-18T09:13:10.307Z
20261118091143,bridge_team_role,fk:role_id,error,0,PASS,,2026-11-18T09:13:17.890Z
20261118091143,bridge_team_role,fk:team_id,error,0,PASS,,2026-11-18T09:13:17.277Z
20261118091143,bridge_team_role,minimum_row_count,error,0,PASS,rows=10,2026-11-18T09:13:15.409Z
20261118091143,bridge_team_role,not_null:role_id,error,0,PASS,,2026-11-18T09:13:16.531Z


status,count
PASS,68


QUANTUM FORCE BRONZE-TO-GOLD BUILD COMPLETED
Run ID: 20261118091143
Gold schema: team_8.quantum_force_gold
Next step: execute the Quantum Force KPI SQL pack.
